## INSTALL

In [ ]:
import sys

REQUIRED_VERSION = (3, 11, 9)
current_version  = sys.version_info[:3]

if current_version != REQUIRED_VERSION:
    raise RuntimeError(
        f"Wrong Python version. Expected 3.11.9, "
        f"got {'.'.join(str(v) for v in current_version)}. "
        "Make sure the correct kernel is selected."
    )

print(f"Python {'.'.join(str(v) for v in current_version)} OK")

%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
%pip install pandas numpy Pillow tqdm scikit-learn xgboost timm lightgbm catboost

## IMPORTS

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
import tqdm
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    StratifiedKFold, train_test_split,
    cross_val_score, ParameterGrid
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from itertools import product

## MODEL SELECTOR

In [ ]:
import os

# Load HF_TOKEN from the .env file in the project root.
# This avoids hardcoding secrets in the notebook.
with open('.env') as env_file:
    for env_line in env_file:
        if '=' in env_line and not env_line.startswith('#'):
            env_key, env_value = env_line.strip().split('=', 1)
            os.environ[env_key] = env_value

print('HF_TOKEN loaded' if os.environ.get('HF_TOKEN') else 'WARNING: HF_TOKEN not found in .env')

In [ ]:
# --- Model Selector -----------------------------------------------------------
# Pick which backbone extracts image features.
# Changing this requires re-running PREPROCESSING and FEATURE EXTRACTION
# to generate a new set of .pt files for the chosen extractor.
#
#   'dinov2'     — DINOv2 ViT-g/14      (Meta, 1536-d)
#                  weights are pre-trained on LVD-142M
#   'convnextv2' — ConvNeXt V2 Huge     (Meta/FAIR, 2816-d)
#                  weights are pre-trained on ImageNet-22k, fine-tuned on ImageNet-1k at 512px
#
FEATURE_EXTRACTOR = 'dinov2'   # 'dinov2' | 'convnextv2'
# -----------------------------------------------------------------------------

## DEFINITIONS

In [ ]:
# --- Device selection ---------------------------------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# --- Load backbone based on FEATURE_EXTRACTOR --------------------------------
# Controlled by the MODEL SELECTOR cell above.
# Each extractor saves its features to files with a unique suffix so you can
# switch models without overwriting previously extracted features.

if FEATURE_EXTRACTOR == 'dinov2':
    # DINOv2 ViT-g/14: 1536-d features, trained on LVD-142M
    backbone_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14')
    backbone_model = backbone_model.to(device).eval()
    backbone_preprocess = transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    def encode(batch):
        return backbone_model(batch.to(device)).float().cpu()

elif FEATURE_EXTRACTOR == 'convnextv2':
    # ConvNeXt V2 Huge: 2816-d features, trained on IN22k then fine-tuned on IN1k at 512px
    import timm
    backbone_model = timm.create_model('convnextv2_huge.fcmae_ft_in22k_in1k_512', pretrained=True, num_classes=0)
    backbone_model = backbone_model.to(device).eval()
    # timm tells us exactly what preprocessing this model expects (crop size, mean, std, etc.)
    convnextv2_data_config = timm.data.resolve_model_data_config(backbone_model)
    backbone_preprocess    = timm.data.create_transform(**convnextv2_data_config, is_training=False)
    def encode(batch):
        return backbone_model(batch.to(device)).float().cpu()

else:
    raise ValueError(f'Unknown FEATURE_EXTRACTOR: {FEATURE_EXTRACTOR!r}. Choose dinov2 or convnextv2.')

# File-name suffix so each extractor saves to its own set of .pt files
FEATURE_SUFFIX_MAP = {'dinov2': '', 'convnextv2': '_convnextv2'}
feat_suffix         = FEATURE_SUFFIX_MAP[FEATURE_EXTRACTOR]
print(f'Backbone: {FEATURE_EXTRACTOR}  |  suffix: {feat_suffix!r}  |  device: {device}')


# --- Image loading and feature extraction helpers ----------------------------

def save_images(image_dir, output_file, has_labels=True):
    # Load every image in a folder, apply backbone_preprocess, and save to a .pt file.
    # has_labels=True  : folder has one subfolder per class
    # has_labels=False : flat folder of images
    image_tensors = []
    labels        = []
    file_paths    = []

    if has_labels:
        class_names    = sorted(os.listdir(image_dir))
        class_to_index = {name: i for i, name in enumerate(class_names)}

        total_images = sum(
            len(os.listdir(os.path.join(image_dir, cls)))
            for cls in class_names
            if os.path.isdir(os.path.join(image_dir, cls))
        )

        count = 0
        for class_name in class_names:
            class_folder = os.path.join(image_dir, class_name)
            for filename in os.listdir(class_folder):
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                image_path = os.path.join(class_folder, filename)
                img        = Image.open(image_path).convert('RGB')
                image_tensors.append(backbone_preprocess(img))
                labels.append(class_to_index[class_name])
                file_paths.append(image_path)
                count += 1
                print(f'{count}/{total_images}', end='\r')

        torch.save({
            'tensors':        torch.stack(image_tensors),
            'labels':         torch.tensor(labels),
            'paths':          file_paths,
            'class_to_index': class_to_index
        }, output_file)

    else:
        image_files  = [
            f for f in os.listdir(image_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        total_images = len(image_files)
        for i, filename in enumerate(image_files, start=1):
            image_path = os.path.join(image_dir, filename)
            img        = Image.open(image_path).convert('RGB')
            image_tensors.append(backbone_preprocess(img))
            file_paths.append(image_path)
            print(f'{i}/{total_images}', end='\r')

        torch.save({'tensors': torch.stack(image_tensors), 'paths': file_paths}, output_file)

    print(f'\nSaved {len(image_tensors)} images to {output_file}')


def extract(input_file, output_file):
    # Pass every image through the selected backbone and save feature vectors.
    data         = torch.load(input_file, weights_only=False)
    images       = data['tensors']
    total_images = len(images)
    feature_list = []

    with torch.no_grad():
        for i, image in enumerate(images, start=1):
            # add batch dimension: (3,H,W) -> (1,3,H,W)
            # encode moves the batch to device and returns a CPU tensor
            feature_vector = encode(image.unsqueeze(0)).squeeze().numpy()
            feature_list.append(feature_vector)
            print(f'{i}/{total_images}', end='\r')

    features = np.array(feature_list)
    torch.save({**data, 'features': features}, output_file)
    print(f'\nExtracted {features.shape} -> saved to {output_file}')

In [ ]:
# --- Metrics helper ----------------------------------------------------------

def compute_classification_metrics(true_labels, predicted_labels):
    # Class distribution shows how balanced the split is, which matters when
    # interpreting accuracy — a 90% accuracy means more on a balanced set.
    unique_classes, samples_per_class = np.unique(true_labels, return_counts=True)

    class_distribution_percentages = {
        int(class_index): round(100 * class_sample_count / len(true_labels), 1)
        for class_index, class_sample_count in zip(unique_classes, samples_per_class)
    }

    return {
        'accuracy':      accuracy_score(true_labels, predicted_labels),
        'f1':            f1_score(true_labels, predicted_labels, average='weighted'),
        'precision':     precision_score(true_labels, predicted_labels, average='weighted'),
        'recall':        recall_score(true_labels, predicted_labels, average='weighted'),
        'class_balance': class_distribution_percentages,
    }


def print_cross_validation_report(model_name, fold_metrics_list):
    # Report mean ± std to capture both performance and stability across folds.
    print(f'--- {model_name} ---')

    for metric_name in ('accuracy', 'f1', 'precision', 'recall'):
        metric_values_per_fold = [fold_result[metric_name] for fold_result in fold_metrics_list]
        mean_value = np.mean(metric_values_per_fold)
        std_value  = np.std(metric_values_per_fold)
        print(f'  {metric_name:<9}: {mean_value:.4f} +/- {std_value:.4f}')

    print('  class balance:')
    for class_index, percentage in fold_metrics_list[0]['class_balance'].items():
        print(f'    class {class_index}: {percentage}%')

    print()


# --- MLP model ---------------------------------------------------------------

class MLP(nn.Module):
    # Configurable fully-connected network for classification on top of pre-extracted features.
    def __init__(self, input_size, num_classes, hidden_layers=(256, 128),
                 activation=nn.ReLU, dropout=0.0):
        super().__init__()

        network_layers     = []
        current_layer_size = input_size

        for layer_output_size in hidden_layers:
            network_layers.append(nn.Linear(current_layer_size, layer_output_size))
            network_layers.append(activation())

            if dropout > 0:
                network_layers.append(nn.Dropout(dropout))

            current_layer_size = layer_output_size

        network_layers.append(nn.Linear(current_layer_size, num_classes))
        self.network = nn.Sequential(*network_layers)

    def forward(self, input_tensor):
        return self.network(input_tensor)


# --- Classifier training functions -------------------------------------------
# All functions accept NumPy arrays and return (trained_model, metrics_dict).

def train_logistic_regression(training_features, training_labels,
                               validation_features, validation_labels,
                               regularisation_strength=1.0):
    model = LogisticRegression(C=regularisation_strength, max_iter=2000)
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_support_vector_machine(training_features, training_labels,
                                  validation_features, validation_labels,
                                  kernel='rbf', regularisation_strength=1.0,
                                  gamma='scale'):
    model = SVC(
        kernel=kernel,
        C=regularisation_strength,
        gamma=gamma,
        decision_function_shape='ovr'
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_random_forest(training_features, training_labels,
                         validation_features, validation_labels,
                         number_of_trees=300):
    model = RandomForestClassifier(
        n_estimators=number_of_trees,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_xgboost_classifier(training_features, training_labels,
                              validation_features, validation_labels):
    # Multi-class softmax objective with log-loss as the evaluation metric
    model = XGBClassifier(
        objective='multi:softmax',
        eval_metric='mlogloss',
        verbosity=0
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_multilayer_perceptron(training_features, training_labels,
                                 validation_features, validation_labels,
                                 hidden_layers=(256, 128), activation=nn.ReLU,
                                 dropout=0.0, epochs=200, learning_rate=1e-3):
    # GPU preferred but CPU works — feature vectors fit comfortably in memory.
    compute_device    = 'cuda' if torch.cuda.is_available() else 'cpu'
    number_of_classes = len(np.unique(training_labels))

    training_input     = torch.tensor(np.asarray(training_features,   np.float32)).to(compute_device)
    training_targets   = torch.tensor(np.asarray(training_labels,     np.int64)).to(compute_device)
    validation_input   = torch.tensor(np.asarray(validation_features, np.float32)).to(compute_device)
    validation_targets = torch.tensor(np.asarray(validation_labels,   np.int64)).to(compute_device)

    training_data_loader = DataLoader(
        TensorDataset(training_input, training_targets),
        batch_size=256,
        shuffle=True
    )

    model         = MLP(training_input.shape[1], number_of_classes, hidden_layers, activation, dropout).to(compute_device)
    optimizer     = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    loss_function = nn.CrossEntropyLoss()

    # Early stopping: track the best validation checkpoint and revert to it at the end
    best_validation_loss       = float('inf')
    best_model_weights         = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()

        for input_batch, target_batch in training_data_loader:
            optimizer.zero_grad()
            batch_loss = loss_function(model(input_batch), target_batch)
            batch_loss.backward()
            optimizer.step()

        model.eval()

        with torch.no_grad():
            current_validation_loss = loss_function(model(validation_input), validation_targets).item()

        if current_validation_loss < best_validation_loss:
            best_validation_loss       = current_validation_loss
            best_model_weights         = model.state_dict()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

            if epochs_without_improvement >= 25:
                break

    model.load_state_dict(best_model_weights)
    model.eval()

    with torch.no_grad():
        predictions = model(validation_input).argmax(dim=1).cpu().numpy()

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )

In [ ]:
def finetune(images_file, train_idx, val_idx, num_classes,
             num_blocks=None, backbone_lr=1e-5, head_lr=1e-3,
             dropout=0.1, epochs=20, batch_size=32,
             patience=5, cache_frozen=True):
    # Fine-tune the last few backbone blocks on your training data.
    # backbone_model must already be loaded (from DEFINITIONS).
    # GPU strongly recommended.
    #
    # Speed / RAM optimisations:
    #   cache_frozen=True  Pre-computes frozen-layer activations once; training only
    #                      runs the small unfrozen suffix each step (~5-10x faster).
    #                      Images are mmap'd (not fully loaded) and freed after caching,
    #                      so peak RAM stays low. Cache stored in fp16 (~5.4 GB for
    #                      convnextv2, ~3 GB for dinov2). Set False if still OOM.
    #   mixed precision    Forward/backward run in fp16 on the GPU (~2x faster).
    #
    # num_blocks=None picks a sensible default per model:
    #   dinov2     : 4 of 40 blocks (~10%)
    #   convnextv2 : 3 of  3 last-stage blocks (the entire final stage)

    print(f'Device: {device}  |  backbone: {FEATURE_EXTRACTOR}  (GPU strongly recommended)')

    # mmap=True: tensors are memory-mapped from disk, not fully loaded into RAM upfront.
    # Pages are read on demand as we iterate through images in the caching loop below,
    # then can be evicted. After caching, we delete the reference to free the pages.
    data   = torch.load(images_file, weights_only=False, mmap=True)
    images = data['tensors']
    labels = data['labels']

    # --- Step 1: Freeze the whole backbone -----------------------------------
    for param in backbone_model.parameters():
        param.requires_grad = False

    # --- Step 2: Identify blocks and norm for this backbone ------------------
    if FEATURE_EXTRACTOR == 'dinov2':
        blocks         = list(backbone_model.blocks)             # 40 transformer blocks total
        norm           = backbone_model.norm
        default_blocks = 4                                       # ~10% of 40
    elif FEATURE_EXTRACTOR == 'convnextv2':
        blocks         = list(backbone_model.stages[-1].blocks)  # last stage: 3 conv blocks
        norm           = backbone_model.norm_pre
        default_blocks = len(blocks)                             # all 3
    if num_blocks is None:
        num_blocks = default_blocks

    # --- Step 3: Unfreeze last num_blocks + norm ------------------------------
    for block in blocks[-num_blocks:]:
        for param in block.parameters():
            param.requires_grad = True
    for param in norm.parameters():
        param.requires_grad = True
    trainable_parameter_count = sum(p.numel() for p in backbone_model.parameters() if p.requires_grad)
    total_parameter_count     = sum(p.numel() for p in backbone_model.parameters())
    print(f'Trainable: {trainable_parameter_count:,}/{total_parameter_count:,} '
          f'({100*trainable_parameter_count/total_parameter_count:.1f}%)  '
          f'[{num_blocks}/{len(blocks)} blocks unfrozen]')

    # --- Step 4: Define frozen prefix and trainable suffix -------------------
    # frozen_prefix : the layers we are NOT updating -- their output never changes.
    # suffix        : the unfrozen layers -- all we need to run per training step.

    def frozen_prefix(batch):
        # Run the backbone up to (but not including) the unfrozen blocks.
        if FEATURE_EXTRACTOR == 'dinov2':
            x = backbone_model.prepare_tokens_with_masks(batch)  # patch embed + pos embed
            for block in backbone_model.blocks[:-num_blocks]:
                x = block(x)
        elif FEATURE_EXTRACTOR == 'convnextv2':
            x = backbone_model.stem(batch)
            for stage in backbone_model.stages[:-1]:              # stages 0-2 (frozen)
                x = stage(x)
            x = backbone_model.stages[-1].downsample(x)           # frozen spatial downsample
        return x

    def suffix(x):
        # Run only the unfrozen layers; return the feature vector.
        if FEATURE_EXTRACTOR == 'dinov2':
            for block in blocks[-num_blocks:]:
                x = block(x)
            x = backbone_model.norm(x)
            return x[:, 0]            # CLS token -> (B, 1536)
        elif FEATURE_EXTRACTOR == 'convnextv2':
            for block in blocks[-num_blocks:]:
                x = block(x)
            x = backbone_model.norm_pre(x)
            return x.mean([-2, -1])   # global average pool -> (B, 2816)

    # --- Step 5: Pre-compute frozen activations (one-time cost) --------------
    # Run all images through frozen_prefix once and cache in fp16.
    # fp16 halves RAM vs fp32 (~5.4 GB instead of ~10.7 GB for convnextv2).
    # After caching, delete image tensors -- training only needs the cache.
    if cache_frozen:
        with torch.no_grad():
            sample_activation = frozen_prefix(images[:1].to(device))
        estimated_cache_ram_gb = sample_activation.numel() * len(images) * 2 / 1e9  # 2 bytes = fp16
        print(f'Caching frozen activations (~{estimated_cache_ram_gb:.1f} GB RAM in fp16)...')
        cached_activation_chunks = []
        backbone_model.eval()
        with torch.no_grad():
            for i in range(0, len(images), batch_size):
                # mmap: only this chunk's pages are in RAM right now
                cached_activation_chunks.append(
                    frozen_prefix(images[i:i+batch_size].to(device)).cpu().half()
                )
                print(f'  {min(i+batch_size, len(images))}/{len(images)}', end='\r')
        frozen_cache = torch.cat(cached_activation_chunks)
        # Free image tensors -- mmap pages can now be evicted, reclaiming RAM
        del cached_activation_chunks, data, images
        print(f'\n  Done. Cache shape: {tuple(frozen_cache.shape)}  (fp16)')
        train_dataset = TensorDataset(frozen_cache[train_idx], labels[train_idx])
        val_source    = frozen_cache[val_idx]
    else:
        # No caching -- run full backbone each step (slower, uses more RAM)
        train_dataset = TensorDataset(images[train_idx], labels[train_idx])
        val_source    = images[val_idx]

    training_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # --- Step 6: Classification head -----------------------------------------
    with torch.no_grad():
        # .float() because this runs outside autocast -- model expects fp32 input
        sample_input      = (frozen_cache if cache_frozen else images)[:1].to(device).float()
        feature_dimension = (suffix(sample_input) if cache_frozen else backbone_model(sample_input)).shape[-1]
    head = nn.Sequential(
        nn.Linear(feature_dimension, 512), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(512, num_classes),
    ).to(device)

    # Backbone params use a much smaller lr to preserve pre-trained knowledge.
    optimizer = torch.optim.AdamW([
        {'params': [p for p in backbone_model.parameters() if p.requires_grad], 'lr': backbone_lr},
        {'params': head.parameters(), 'lr': head_lr},
    ], weight_decay=1e-4)
    loss_function = nn.CrossEntropyLoss()

    # Mixed precision: matmuls run in fp16 -- ~2x faster, no meaningful accuracy loss.
    is_mixed_precision_enabled = (device == 'cuda')
    gradient_scaler            = torch.amp.GradScaler('cuda', enabled=is_mixed_precision_enabled)
    # Load cache slices as fp16 on GPU (matches autocast); fp32 on CPU fallback.
    batch_load_dtype = torch.float16 if is_mixed_precision_enabled else torch.float32

    trainable_param_names = {name for name, p in backbone_model.named_parameters() if p.requires_grad}

    # Helper: run source (cached acts or raw images) through the right forward fn.
    # Always uses fp32 since it runs outside the autocast training context.
    def extract_features_from_source(source):
        feature_chunks = []
        with torch.no_grad():
            for i in range(0, len(source), batch_size):
                chunk = source[i:i+batch_size].to(device=device, dtype=torch.float32)
                feature_chunks.append(
                    (suffix(chunk) if cache_frozen else backbone_model(chunk)).cpu()
                )
        return torch.cat(feature_chunks)

    # --- Step 7: Training loop with early stopping ---------------------------
    best_validation_loss           = float('inf')
    best_backbone_state            = None
    best_classification_head_state = None
    epochs_without_improvement     = 0

    for epoch in range(epochs):
        backbone_model.train()
        head.train()
        for input_batch, target_batch in training_data_loader:
            # fp16 on GPU (inside autocast), fp32 on CPU
            input_batch  = input_batch.to(device=device, dtype=batch_load_dtype)
            target_batch = target_batch.to(device)
            optimizer.zero_grad()
            with torch.autocast('cuda', enabled=is_mixed_precision_enabled):  # fp16 forward + backward
                batch_features = suffix(input_batch) if cache_frozen else backbone_model(input_batch)
                batch_loss     = loss_function(head(batch_features), target_batch)
            gradient_scaler.scale(batch_loss).backward()
            gradient_scaler.step(optimizer)
            gradient_scaler.update()

        backbone_model.eval()
        head.eval()
        validation_features     = extract_features_from_source(val_source).to(device)
        current_validation_loss = loss_function(
            head(validation_features), labels[val_idx].to(device)
        ).item()

        print(f'Epoch {epoch+1:>3}/{epochs}  val_loss={current_validation_loss:.4f}', end='\r')

        if current_validation_loss < best_validation_loss:
            best_validation_loss = current_validation_loss
            # Only save the unfrozen params -- avoids duplicating the whole model
            best_backbone_state = {
                k: v.cpu().clone()
                for k, v in backbone_model.state_dict().items()
                if k in trainable_param_names
            }
            best_classification_head_state = {
                k: v.cpu().clone() for k, v in head.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break

    print(f'\nBest val_loss: {best_validation_loss:.4f}')

    # --- Step 8: Restore best weights ----------------------------------------
    current_backbone_state = backbone_model.state_dict()
    current_backbone_state.update({k: v.to(device) for k, v in best_backbone_state.items()})
    backbone_model.load_state_dict(current_backbone_state)
    head.load_state_dict({k: v.to(device) for k, v in best_classification_head_state.items()})

    # Final val predictions with restored weights
    backbone_model.eval()
    head.eval()
    validation_predictions = (
        head(extract_features_from_source(val_source).to(device)).argmax(dim=1).cpu().numpy()
    )

    # Re-freeze backbone so subsequent extract() calls behave correctly
    for param in backbone_model.parameters():
        param.requires_grad = False

    return head, compute_classification_metrics(labels[val_idx].numpy(), validation_predictions)

In [ ]:
def train_knn_classifier(training_features, training_labels,
                          validation_features, validation_labels,
                          number_of_neighbours=10, distance_metric='cosine'):
    # Cosine distance outperforms Euclidean for high-dimensional backbone embeddings.
    # DINOv2 and ConvNeXt features lie on roughly spherical manifolds, so
    # angular separation is more discriminative than absolute magnitude differences.
    model = KNeighborsClassifier(
        n_neighbors=number_of_neighbours,
        metric=distance_metric,
        n_jobs=-1
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_lightgbm_classifier(training_features, training_labels,
                               validation_features, validation_labels,
                               number_of_estimators=500, max_leaf_nodes=63,
                               learning_rate=0.05):
    # Histogram-based gradient boosting — typically faster than XGBoost
    # with comparable accuracy on tabular feature data.
    model = LGBMClassifier(
        n_estimators=number_of_estimators,
        num_leaves=max_leaf_nodes,
        learning_rate=learning_rate,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_catboost_classifier(training_features, training_labels,
                               validation_features, validation_labels,
                               number_of_iterations=500, tree_depth=6,
                               learning_rate=0.05):
    # Ordered boosting with strong defaults — less hand-tuning needed than XGBoost.
    # CatBoost predict() returns float, so we cast to int for metric compatibility.
    model = CatBoostClassifier(
        iterations=number_of_iterations,
        depth=tree_depth,
        learning_rate=learning_rate,
        verbose=0,
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64)).astype(int)

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_stacking_ensemble(training_features, training_labels,
                             validation_features, validation_labels):
    # Stacking: base learners generate out-of-fold meta-features via 5-fold CV,
    # then a LogisticRegression meta-learner classifies on those meta-features.
    # Typically +0.5–1% F1 over the best individual model.
    #
    # Base learners use n_jobs=1 to prevent nested parallelism deadlocks when
    # StackingClassifier already parallelises across CV folds with n_jobs=-1.
    base_estimators = [
        ('svm',      SVC(kernel='rbf', probability=True, decision_function_shape='ovr')),
        ('lightgbm', LGBMClassifier(n_estimators=300, num_leaves=63, verbose=-1, n_jobs=1)),
        ('knn',      KNeighborsClassifier(n_neighbors=10, metric='cosine', n_jobs=1)),
    ]

    model = StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(max_iter=2000),
        cv=5,
        n_jobs=-1,
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )

In [ ]:
# These helpers are used by the Feature Engineering cell below.

def image_id(file_path, is_train):
    # Extract the image ID from a file path.
    stem = os.path.splitext(os.path.basename(file_path))[0]
    return f'train_{stem}' if is_train else stem


def load_csv(csv_path, image_ids):
    # Load a CSV (indexed by image_id) and return rows in the same order as image_ids.
    # This ensures the CSV rows line up with our feature matrix rows.
    df = pd.read_csv(csv_path, index_col='image_id')
    return df.loc[image_ids].to_numpy(dtype=np.float32)


def reduce_hist(histogram, factor):
    # Merge every 'factor' consecutive bins by averaging.
    if factor == 1:
        return histogram
    num_bins = histogram.shape[1]
    if num_bins % factor != 0:
        valid = [f for f in range(1, num_bins + 1) if num_bins % f == 0]
        raise ValueError(
            f'factor {factor} does not divide evenly into {num_bins} bins. '
            f'Valid factors: {valid}'
        )
    return histogram.reshape(histogram.shape[0], num_bins // factor, factor).mean(axis=2)

## PREPROCESSING
Load images, extract features for both backbones, save to disk. Only needs to run once.

In [ ]:
train_image_dir = r'Data\task2_data\images\train'
test_image_dir  = r'Data\task2_data\images\test'

if not os.path.exists(train_image_dir):
    raise FileNotFoundError(f'Training folder not found: {train_image_dir}')
if not os.path.exists(test_image_dir):
    raise FileNotFoundError(f'Test folder not found: {test_image_dir}')

# Save preprocessed tensors for the currently selected backbone.
# The feat_suffix keeps dinov2 and convnextv2 files separate.
print('Saving preprocessed image tensors...')
save_images(train_image_dir, f'Data\\task2_data\\t2_train{feat_suffix}.pt', has_labels=True)
save_images(test_image_dir,  f'Data\\task2_data\\t2_test{feat_suffix}.pt',  has_labels=False)

# --- Extract features for both DINOv2 and ConvNeXt V2 in one pass -----------
# Each model loads images from disk with its own preprocessing so features are
# always correct. Saves separate feature files; downstream cells pick the right
# one via feat_suffix.

import timm

compute_device = 'cuda' if torch.cuda.is_available() else 'cpu'

# DINOv2 preprocessing (standard ImageNet stats, 224px center crop)
dinov2_image_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ConvNeXt V2 preprocessing -- timm resolves the exact crop size and norm for this checkpoint
convnextv2_temp_model      = timm.create_model('convnextv2_huge.fcmae_ft_in22k_in1k_512', pretrained=False, num_classes=0)
convnextv2_data_config     = timm.data.resolve_model_data_config(convnextv2_temp_model)
convnextv2_image_transform = timm.data.create_transform(**convnextv2_data_config, is_training=False)
del convnextv2_temp_model  # only needed to read the config

# Each entry: (backbone_name, file_suffix, image_transform, load_backbone_fn, encode_fn)
BACKBONE_CONFIGS = [
    ('dinov2',     '',            dinov2_image_transform,
     lambda: torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14'),
     lambda model, batch: model(batch).float()),
    ('convnextv2', '_convnextv2', convnextv2_image_transform,
     lambda: timm.create_model('convnextv2_huge.fcmae_ft_in22k_in1k_512', pretrained=True, num_classes=0),
     lambda model, batch: model(batch).float()),
]


def load_images_from_dir(image_dir, image_transform, has_labels):
    # Load every image from image_dir, apply image_transform.
    # Returns (image_tensors, label_tensor_or_None, file_paths, class_to_index_dict).
    image_tensors_list = []
    label_list         = []
    file_paths         = []
    class_to_index     = {}

    if has_labels:
        class_names  = sorted(os.listdir(image_dir))
        class_to_index = {cls: i for i, cls in enumerate(class_names)}
        total_images = sum(
            len(os.listdir(os.path.join(image_dir, cls)))
            for cls in class_names
            if os.path.isdir(os.path.join(image_dir, cls))
        )
        image_count = 0
        for class_name in class_names:
            for filename in os.listdir(os.path.join(image_dir, class_name)):
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                image_path = os.path.join(image_dir, class_name, filename)
                image_tensors_list.append(image_transform(Image.open(image_path).convert('RGB')))
                label_list.append(class_to_index[class_name])
                file_paths.append(image_path)
                image_count += 1
                print(f'  {image_count}/{total_images}', end='\r')
    else:
        image_files  = sorted(
            fn for fn in os.listdir(image_dir)
            if fn.lower().endswith(('.jpg', '.jpeg', '.png'))
        )
        for i, filename in enumerate(image_files, 1):
            image_path = os.path.join(image_dir, filename)
            image_tensors_list.append(image_transform(Image.open(image_path).convert('RGB')))
            file_paths.append(image_path)
            print(f'  {i}/{len(image_files)}', end='\r')

    label_tensor = torch.tensor(label_list) if label_list else None
    return torch.stack(image_tensors_list), label_tensor, file_paths, class_to_index


def extract_and_save_features(backbone_model_instance, encode_function, image_tensors,
                               label_tensor, file_paths, class_to_index, output_file):
    # Run image_tensors through backbone_model_instance and save features + metadata.
    feature_vectors = []
    with torch.no_grad():
        for i, tensor in enumerate(image_tensors, 1):
            feature_vectors.append(
                encode_function(backbone_model_instance, tensor.unsqueeze(0).to(compute_device))
                .squeeze().cpu().numpy()
            )
            print(f'  {i}/{len(image_tensors)}', end='\r')

    save_dict = {
        'tensors':  image_tensors,
        'paths':    file_paths,
        'features': np.array(feature_vectors),
    }
    if label_tensor is not None:
        save_dict['labels']         = label_tensor
        save_dict['class_to_index'] = class_to_index
    torch.save(save_dict, output_file)
    print(f'\n  Saved {np.array(feature_vectors).shape} -> {output_file}')


# Main extraction loop ---------------------------------------------------------
for backbone_name, backbone_suffix, image_transform, load_backbone_fn, encode_fn in BACKBONE_CONFIGS:
    print(f'\n=== {backbone_name} (device={compute_device}) ===')
    backbone_model_instance = load_backbone_fn()
    backbone_model_instance = backbone_model_instance.to(compute_device).eval()

    print('  train images:')
    train_image_tensors, train_label_tensors, train_file_paths, train_class_to_index = \
        load_images_from_dir(train_image_dir, image_transform, has_labels=True)
    extract_and_save_features(
        backbone_model_instance, encode_fn,
        train_image_tensors, train_label_tensors, train_file_paths, train_class_to_index,
        f'Data\\task2_data\\t2_train_features{backbone_suffix}.pt'
    )

    print('  test images:')
    test_image_tensors, _, test_file_paths, _ = \
        load_images_from_dir(test_image_dir, image_transform, has_labels=False)
    extract_and_save_features(
        backbone_model_instance, encode_fn,
        test_image_tensors, None, test_file_paths, {},
        f'Data\\task2_data\\t2_test_features{backbone_suffix}.pt'
    )

    del backbone_model_instance
    if compute_device == 'cuda':
        torch.cuda.empty_cache()

print('\nAll done -- feature files ready for dinov2 / convnextv2.')

## FINE-TUNING
Optionally fine-tune the last few backbone blocks on your data, then re-extract features.
Supports both backbones: `dinov2` and `convnextv2`.
Skip this section if you want to use frozen features only.

In [ ]:
# Fine-tune the last few backbone blocks on task 2 training images.
# After this runs, re-extract features from the adapted backbone so the rest of
# the notebook (PCA, EVAL, HYPER SEARCH) automatically benefits.
#
# GPU strongly recommended - on CPU this will take a long time.
#
# IMPORTANT: The PREPROCESSING cell must have been run with the current
# FEATURE_EXTRACTOR selected so that t2_train{feat_suffix}.pt uses the correct preprocessing.

raw_pt      = torch.load(f'Data\\task2_data\\t2_train{feat_suffix}.pt', weights_only=False)
num_classes = len(np.unique(raw_pt['labels'].numpy()))
all_idx     = np.arange(len(raw_pt['labels']))
train_idx_ft, val_idx_ft = train_test_split(
    all_idx, test_size=0.2, random_state=42,
    stratify=raw_pt['labels'].numpy()
)

fine_tuned_head, ft_metrics = finetune(
    images_file = f'Data\\task2_data\\t2_train{feat_suffix}.pt',
    train_idx   = train_idx_ft,
    val_idx     = val_idx_ft,
    num_classes = num_classes,
    num_blocks  = 2,        # fewer blocks for the smaller dataset
    backbone_lr = 5e-6,
    head_lr     = 1e-3,
    dropout     = 0.2,
    epochs      = 15,
    batch_size  = 8,        # keep small — see task1 comment above
    patience    = 7,
)

print(f'Fine-tuned val:  F1={ft_metrics["f1"]:.4f}  Acc={ft_metrics["accuracy"]:.4f}')

# Re-extract features using the now-adapted backbone and save to new files.
# In the PCA cell, set USE_FINETUNED = True to use these.
print('\nRe-extracting features with fine-tuned backbone...')
extract(f'Data\\task2_data\\t2_train{feat_suffix}.pt', f'Data\\task2_data\\t2_train_features_ft{feat_suffix}.pt')
extract(f'Data\\task2_data\\t2_test{feat_suffix}.pt',  f'Data\\task2_data\\t2_test_features_ft{feat_suffix}.pt')
print('Done. Set USE_FINETUNED = True in the PCA cell to use these features.')

## PCA
Reduce the 1536-d DINOv2 features to a smaller number of dimensions.

In [ ]:
# Control Board ------
USE_FINETUNED = True
USE_PCA = False
# NOTE: Fine-tuned features (USE_FINETUNED=True) only exist for FEATURE_EXTRACTOR='dinov2'
# --------------------

if USE_FINETUNED:
    train_data = torch.load(f'Data\\task2_data\\t2_train_features_ft{feat_suffix}.pt', weights_only=False)
    test_data  = torch.load(f'Data\\task2_data\\t2_test_features_ft{feat_suffix}.pt',  weights_only=False)
else:
    train_data = torch.load(f'Data\\task2_data\\t2_train_features{feat_suffix}.pt', weights_only=False)
    test_data  = torch.load(f'Data\\task2_data\\t2_test_features{feat_suffix}.pt',  weights_only=False)

raw_train_features = train_data['features']
train_labels       = train_data['labels'].numpy()
raw_test_features  = test_data['features']

print(f'Train: {raw_train_features.shape}  Test: {raw_test_features.shape}')
print(f'Classes: {train_data["class_to_index"]}')

# --- Plot variance explained to pick the right number of PCA components ------
# Always plot even if USE_PCA = False, so you can see what you would be reducing to.
pca_full = PCA(random_state=42)
pca_full.fit(raw_train_features)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

n_50 = int(np.searchsorted(cumulative_variance, 0.50)) + 1
n_75 = int(np.searchsorted(cumulative_variance, 0.75)) + 1
n_90 = int(np.searchsorted(cumulative_variance, 0.90)) + 1
n_95 = int(np.searchsorted(cumulative_variance, 0.95)) + 1
n_99 = int(np.searchsorted(cumulative_variance, 0.99)) + 1

plt.figure(figsize=(9, 4))
plt.plot(cumulative_variance, linewidth=1.5)
plt.axhline(0.50, color='blue',   linestyle='--', label=f'50% ({n_50} components)')
plt.axhline(0.75, color='green',  linestyle='--', label=f'75% ({n_75} components)')
plt.axhline(0.90, color='purple', linestyle='--', label=f'90% ({n_90} components)')
plt.axhline(0.95, color='orange', linestyle='--', label=f'95% ({n_95} components)')
plt.axhline(0.99, color='red',    linestyle='--', label=f'99% ({n_99} components)')
plt.xlabel('Number of PCA components')
plt.ylabel('Cumulative variance explained')
plt.title(f'PCA - Task 2 {FEATURE_EXTRACTOR} features ({raw_train_features.shape[1]}-d)')
plt.legend()
plt.tight_layout()
plt.show()

print('Components needed:')
for label, n in [('50%', n_50), ('75%', n_75), ('90%', n_90), ('95%', n_95), ('99%', n_99)]:
    print(f'  {label} variance -> {n} components')

# --- Apply PCA -------------------------------------------------------------------
NUM_PCA_COMPONENTS = n_99   # change to n_50 / n_75 / n_90 / n_95 / n_99 as desired

if USE_PCA:
    pca = PCA(n_components=NUM_PCA_COMPONENTS, random_state=42)
    pca_train_features = pca.fit_transform(raw_train_features)
    pca_test_features  = pca.transform(raw_test_features)
    print(f'\nPCA: {raw_train_features.shape[1]}-d -> {pca_train_features.shape[1]}-d')
    print(f'Variance retained: {pca.explained_variance_ratio_.sum():.4f}')
else:
    pca_train_features = raw_train_features
    pca_test_features  = raw_test_features
    print(f'\nPCA skipped - using full {pca_train_features.shape[1]}-d features')

## FEATURE ENGINEERING
Combine ResNet PCA features with colour histograms, HOG, and extra features.

In [ ]:
# --- Feature toggles ---------------------------------------------------------
USE_COLOR_HISTOGRAM     = True   # 96-d colour distribution
USE_HOG_FEATURES        = True   # 100-d HOG shape/edge features
USE_EXTRA_FEATURES      = True   # 23-d features

# --- Options -----------------------------------------------------------------
# 96 bins total so valid factors: 1 (keep all), 2->48d, 3->32d, 4->24d, 6->16d, 8->12d
HISTOGRAM_REDUCE_FACTOR = 6

# PCA on the extra features. None = keep all 23 standardised dims.
EXTRA_FEATURES_PCA_DIMS = 12
# -----------------------------------------------------------------------------

# Build the image ID lists so we can match CSV rows to feature matrix rows
train_image_ids = [image_id(p, is_train=True)  for p in train_data['paths']]
test_image_ids  = [image_id(p, is_train=False) for p in test_data['paths']]

# Start with the PCA-reduced features (kept as-is, not rescaled).
combined_train_features = pca_train_features.copy()
combined_test_features  = pca_test_features.copy()

print(f'         PCA features : {pca_train_features.shape[1]:>4d}-d')

if USE_COLOR_HISTOGRAM:
    col_train = load_csv(r'Data\task2_data\color_histogram.csv', train_image_ids)
    col_test  = load_csv(r'Data\task2_data\color_histogram.csv', test_image_ids)
    col_train = reduce_hist(col_train, HISTOGRAM_REDUCE_FACTOR)
    col_test  = reduce_hist(col_test,  HISTOGRAM_REDUCE_FACTOR)

    # standardise: fit only on training data to avoid leaking test info
    scaler = StandardScaler().fit(col_train)
    combined_train_features = np.hstack([combined_train_features, scaler.transform(col_train)])
    combined_test_features  = np.hstack([combined_test_features,  scaler.transform(col_test)])
    note = f'factor={HISTOGRAM_REDUCE_FACTOR}' if HISTOGRAM_REDUCE_FACTOR > 1 else 'full 96 bins'
    print(f'+ colour histogram  : {col_train.shape[1]:>4d}-d  (standardised, {note})')

if USE_HOG_FEATURES:
    hog_train = load_csv(r'Data\task2_data\hog_pca.csv', train_image_ids)
    hog_test  = load_csv(r'Data\task2_data\hog_pca.csv', test_image_ids)

    # HOG is already PCA-reduced so we keep it as-is
    combined_train_features = np.hstack([combined_train_features, hog_train])
    combined_test_features  = np.hstack([combined_test_features,  hog_test])
    print(f'+ HOG features      : {hog_train.shape[1]:>4d}-d  (as-is)')

if USE_EXTRA_FEATURES:
    extra_train = load_csv(r'Data\task2_data\additional_features.csv', train_image_ids)
    extra_test  = load_csv(r'Data\task2_data\additional_features.csv', test_image_ids)

    # standardise first (fit on train only)
    scaler = StandardScaler().fit(extra_train)
    extra_train_scaled = scaler.transform(extra_train)
    extra_test_scaled  = scaler.transform(extra_test)

    if EXTRA_FEATURES_PCA_DIMS is not None:
        pca_extra          = PCA(n_components=EXTRA_FEATURES_PCA_DIMS, random_state=42)
        extra_train_scaled = pca_extra.fit_transform(extra_train_scaled)
        extra_test_scaled  = pca_extra.transform(extra_test_scaled)
        var_kept = pca_extra.explained_variance_ratio_.sum()
        print(f'+ extra features    : {extra_train_scaled.shape[1]:>4d}-d  (standardised -> PCA, {var_kept:.1%} variance kept)')
    else:
        print(f'+ extra features    : {extra_train_scaled.shape[1]:>4d}-d  (standardised)')

    combined_train_features = np.hstack([combined_train_features, extra_train_scaled])
    combined_test_features  = np.hstack([combined_test_features,  extra_test_scaled])

print(f'{"-"*40}')
print(f'  Total               : {combined_train_features.shape[1]:>4d}-d  ({combined_train_features.shape[0]} train, {combined_test_features.shape[0]} test)')

## MODEL EVALUATION

In [ ]:
# --- Control Board -----------------------------------------------------------
SVM_KERNEL             = 'rbf'
KNN_NEAREST_NEIGHBOURS = 15
RANDOM_FOREST_NUM_TREES = 300
# -----------------------------------------------------------------------------

cross_validation_splitter  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
number_of_features         = combined_train_features.shape[1]
print(f'Running 5-fold CV on {number_of_features}-d features, {len(train_labels)} samples...\n')

# Logistic Regression
fold_metrics_list = []
for train_fold_idx, val_fold_idx in cross_validation_splitter.split(combined_train_features, train_labels):
    _, metrics = train_logistic_regression(
        combined_train_features[train_fold_idx], train_labels[train_fold_idx],
        combined_train_features[val_fold_idx],   train_labels[val_fold_idx]
    )
    fold_metrics_list.append(metrics)
print_cross_validation_report('Logistic Regression (5-fold CV)', fold_metrics_list)

# SVM
fold_metrics_list = []
for train_fold_idx, val_fold_idx in cross_validation_splitter.split(combined_train_features, train_labels):
    _, metrics = train_support_vector_machine(
        combined_train_features[train_fold_idx], train_labels[train_fold_idx],
        combined_train_features[val_fold_idx],   train_labels[val_fold_idx],
        kernel=SVM_KERNEL
    )
    fold_metrics_list.append(metrics)
print_cross_validation_report(f'SVM kernel={SVM_KERNEL} (5-fold CV)', fold_metrics_list)

# kNN (cosine distance)
fold_metrics_list = []
for train_fold_idx, val_fold_idx in cross_validation_splitter.split(combined_train_features, train_labels):
    _, metrics = train_knn_classifier(
        combined_train_features[train_fold_idx], train_labels[train_fold_idx],
        combined_train_features[val_fold_idx],   train_labels[val_fold_idx],
        number_of_neighbours=KNN_NEAREST_NEIGHBOURS,
        distance_metric='cosine'
    )
    fold_metrics_list.append(metrics)
print_cross_validation_report(f'kNN k={KNN_NEAREST_NEIGHBOURS} cosine (5-fold CV)', fold_metrics_list)

# Random Forest
fold_metrics_list = []
for train_fold_idx, val_fold_idx in cross_validation_splitter.split(combined_train_features, train_labels):
    _, metrics = train_random_forest(
        combined_train_features[train_fold_idx], train_labels[train_fold_idx],
        combined_train_features[val_fold_idx],   train_labels[val_fold_idx],
        number_of_trees=RANDOM_FOREST_NUM_TREES
    )
    fold_metrics_list.append(metrics)
print_cross_validation_report(f'Random Forest n_trees={RANDOM_FOREST_NUM_TREES} (5-fold CV)', fold_metrics_list)

# XGBoost
fold_metrics_list = []
for train_fold_idx, val_fold_idx in cross_validation_splitter.split(combined_train_features, train_labels):
    _, metrics = train_xgboost_classifier(
        combined_train_features[train_fold_idx], train_labels[train_fold_idx],
        combined_train_features[val_fold_idx],   train_labels[val_fold_idx]
    )
    fold_metrics_list.append(metrics)
print_cross_validation_report('XGBoost (5-fold CV)', fold_metrics_list)

# LightGBM
fold_metrics_list = []
for train_fold_idx, val_fold_idx in cross_validation_splitter.split(combined_train_features, train_labels):
    _, metrics = train_lightgbm_classifier(
        combined_train_features[train_fold_idx], train_labels[train_fold_idx],
        combined_train_features[val_fold_idx],   train_labels[val_fold_idx]
    )
    fold_metrics_list.append(metrics)
print_cross_validation_report('LightGBM (5-fold CV)', fold_metrics_list)

# CatBoost
fold_metrics_list = []
for train_fold_idx, val_fold_idx in cross_validation_splitter.split(combined_train_features, train_labels):
    _, metrics = train_catboost_classifier(
        combined_train_features[train_fold_idx], train_labels[train_fold_idx],
        combined_train_features[val_fold_idx],   train_labels[val_fold_idx]
    )
    fold_metrics_list.append(metrics)
print_cross_validation_report('CatBoost (5-fold CV)', fold_metrics_list)

# Stacking (SVM + LightGBM + kNN -> LogReg meta-learner)
# Note: slower than the others -- trains 3 base models via 5-fold CV
fold_metrics_list = []
for train_fold_idx, val_fold_idx in cross_validation_splitter.split(combined_train_features, train_labels):
    _, metrics = train_stacking_ensemble(
        combined_train_features[train_fold_idx], train_labels[train_fold_idx],
        combined_train_features[val_fold_idx],   train_labels[val_fold_idx]
    )
    fold_metrics_list.append(metrics)
print_cross_validation_report('Stacking SVM+LightGBM+kNN -> LogReg (5-fold CV)', fold_metrics_list)

## HYPER PARAM SEARCH

In [ ]:
import json

# --- Search grids ------------------------------------------------------------
# Feature engineering options (n_50, n_75, n_90, n_95, n_99 from the PCA cell)
# None = skip PCA entirely and use the full 1536-d DINOv2 features
SEARCH_PCA_DIMS    = [None, n_90, n_95, n_99]
SEARCH_HIST_FACTOR = [1, 2, 6]
SEARCH_EXTRA_PCA   = [None, 10, 15]

SEARCH_LOGREG = [{'C': 0.1}, {'C': 1}, {'C': 10}]

SEARCH_SVM = [
    {'kernel': 'linear', 'C': 0.1},
    {'kernel': 'linear', 'C': 1},
    {'kernel': 'linear', 'C': 10},
    {'kernel': 'rbf',    'C': 1,   'gamma': 'scale'},
    {'kernel': 'rbf',    'C': 10,  'gamma': 'scale'},
    {'kernel': 'rbf',    'C': 100, 'gamma': 'scale'},
]

SEARCH_KNN = [
    {'k': 5,  'metric': 'cosine'},
    {'k': 10, 'metric': 'cosine'},
    {'k': 15, 'metric': 'cosine'},
    {'k': 10, 'metric': 'euclidean'},
]

SEARCH_RF = [
    {'n_trees': 200},
    {'n_trees': 300},
]

SEARCH_XGB = [
    {'n_estimators': 100, 'max_depth': 6,  'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 4,  'learning_rate': 0.1},
]

SEARCH_LGBM = [
    {'n_estimators': 200, 'num_leaves': 31,  'learning_rate': 0.1},
    {'n_estimators': 500, 'num_leaves': 63,  'learning_rate': 0.05},
    {'n_estimators': 500, 'num_leaves': 127, 'learning_rate': 0.05},
]

SEARCH_CATBOOST = [
    {'iterations': 200, 'depth': 4, 'learning_rate': 0.1},
    {'iterations': 500, 'depth': 6, 'learning_rate': 0.05},
]

SEARCH_MLP = [
    {'hidden_layers': (128, 64), 'activation': nn.GELU, 'dropout': 0.2, 'learning_rate': 1e-3},
    {'hidden_layers': (64, 32),  'activation': nn.GELU, 'dropout': 0.2, 'learning_rate': 1e-3},
]
# -----------------------------------------------------------------------------

# Pre-load CSVs once to avoid reading disk repeatedly
color_histogram_train_data = load_csv(r'Data\task2_data\color_histogram.csv',     train_image_ids)
hog_features_train_data    = load_csv(r'Data\task2_data\hog_pca.csv',             train_image_ids)
extra_features_train_data  = load_csv(r'Data\task2_data\additional_features.csv', train_image_ids)

# Pre-compute PCA for each unique dimension count.
print('Pre-computing PCA projections...')
pca_features_cache = {}

for num_pca_dimensions in SEARCH_PCA_DIMS:
    if num_pca_dimensions is None:
        pca_features_cache[None] = raw_train_features.astype(np.float64)
        print(f'  None (no PCA): {raw_train_features.shape[1]}-d')
    else:
        pca_reducer = PCA(n_components=num_pca_dimensions, random_state=42)
        pca_features_cache[num_pca_dimensions] = pca_reducer.fit_transform(raw_train_features).astype(np.float64)
        print(f'  {num_pca_dimensions} components: done')

# Use a fixed train/val split (first fold of 5-fold CV)
train_idx, val_idx = next(
    StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(raw_train_features, train_labels)
)

print(f'\nTrain: {len(train_idx)} samples  Val: {len(val_idx)} samples')

def build_combined_features(num_pca_dims, histogram_factor, extra_pca_dims):
    # Assemble the combined feature matrix for a given set of options.
    combined_feature_matrix = pca_features_cache[num_pca_dims]

    reduced_color_histogram = reduce_hist(color_histogram_train_data.copy(), histogram_factor)
    color_histogram_scaler  = StandardScaler().fit(reduced_color_histogram[train_idx])
    combined_feature_matrix = np.hstack([
        combined_feature_matrix,
        color_histogram_scaler.transform(reduced_color_histogram)
    ])

    combined_feature_matrix = np.hstack([
        combined_feature_matrix,
        hog_features_train_data.astype(np.float64)
    ])

    extra_features_scaler  = StandardScaler().fit(extra_features_train_data[train_idx])
    extra_features_scaled  = extra_features_scaler.transform(extra_features_train_data)

    if extra_pca_dims is not None:
        extra_features_pca_reducer = PCA(n_components=extra_pca_dims, random_state=42)
        extra_features_pca_reducer.fit(extra_features_scaled[train_idx])
        extra_features_scaled = extra_features_pca_reducer.transform(extra_features_scaled)

    combined_feature_matrix = np.hstack([combined_feature_matrix, extra_features_scaled])

    return combined_feature_matrix


# Pre-build all feature matrices
print('\nBuilding feature matrices...')
all_feature_configurations = list(product(SEARCH_PCA_DIMS, SEARCH_HIST_FACTOR, SEARCH_EXTRA_PCA))
prebuilt_feature_matrices  = {}

for feature_config in all_feature_configurations:
    prebuilt_feature_matrices[feature_config] = build_combined_features(*feature_config)

print(f'  {len(all_feature_configurations)} feature configs ready')

num_model_configs = (len(SEARCH_LOGREG) + len(SEARCH_SVM) + len(SEARCH_KNN)
                     + len(SEARCH_RF) + len(SEARCH_XGB)
                     + len(SEARCH_LGBM) + len(SEARCH_CATBOOST) + len(SEARCH_MLP))

total_evaluations = len(all_feature_configurations) * num_model_configs
print(f'\n{len(all_feature_configurations)} feature configs x {num_model_configs} model configs = {total_evaluations} total\n')

search_results = []

def record_search_result(model_name, param_description, params_json_str,
                         pca_dims, hist_factor, extra_pca, total_dims, validation_f1):
    search_results.append({
        'model':       model_name,
        'params':      param_description,
        'params_json': params_json_str,
        'pca_dims':    str(pca_dims) if pca_dims is None else pca_dims,
        'hist_factor': hist_factor,
        'extra_pca':   str(extra_pca),
        'total_dims':  total_dims,
        'f1':          float(validation_f1),
    })

progress_bar = tqdm.tqdm(total=total_evaluations, desc='Searching')

# --- Pass 1: Logistic Regression, SVM, kNN (fast) ----------------------------
progress_bar.set_description('Pass 1/3 - LogReg, SVM, kNN')
for pca_dims, hist_factor, extra_pca_dims in all_feature_configurations:

    feature_matrix           = prebuilt_feature_matrices[(pca_dims, hist_factor, extra_pca_dims)]
    train_split_features     = feature_matrix[train_idx]
    val_split_features       = feature_matrix[val_idx]
    train_split_labels       = train_labels[train_idx]
    val_split_labels         = train_labels[val_idx]
    total_feature_dimensions = feature_matrix.shape[1]

    # Logistic Regression
    for params in SEARCH_LOGREG:
        model = LogisticRegression(max_iter=2000, C=params['C'])
        model.fit(train_split_features, train_split_labels)
        validation_f1 = f1_score(val_split_labels, model.predict(val_split_features), average='weighted')
        record_search_result('LogReg', f"C={params['C']}", json.dumps({'C': params['C']}),
                             pca_dims, hist_factor, extra_pca_dims, total_feature_dimensions, validation_f1)
        progress_bar.update(1)

    # SVM
    for params in SEARCH_SVM:
        extra_kwargs = {k: v for k, v in params.items() if k != 'kernel'}
        model        = SVC(kernel=params['kernel'], decision_function_shape='ovr', **extra_kwargs)
        model.fit(train_split_features, train_split_labels)
        validation_f1 = f1_score(val_split_labels, model.predict(val_split_features), average='weighted')
        gamma_str = f" gamma={params['gamma']}" if 'gamma' in params else ''
        record_search_result('SVM', f"kernel={params['kernel']} C={params['C']}{gamma_str}",
                             json.dumps(params), pca_dims, hist_factor, extra_pca_dims,
                             total_feature_dimensions, validation_f1)
        progress_bar.update(1)

    # kNN (cosine and euclidean)
    for params in SEARCH_KNN:
        model = KNeighborsClassifier(n_neighbors=params['k'], metric=params['metric'], n_jobs=-1)
        model.fit(train_split_features, train_split_labels)
        validation_f1 = f1_score(val_split_labels, model.predict(val_split_features), average='weighted')
        record_search_result('kNN', f"k={params['k']} metric={params['metric']}",
                             json.dumps({'k': params['k'], 'metric': params['metric']}),
                             pca_dims, hist_factor, extra_pca_dims, total_feature_dimensions, validation_f1)
        progress_bar.update(1)

# --- Pass 2: Random Forest, XGBoost, LightGBM, CatBoost (medium) -------------
progress_bar.set_description('Pass 2/3 - RF, XGB, LGBM, CatBoost')
for pca_dims, hist_factor, extra_pca_dims in all_feature_configurations:

    feature_matrix           = prebuilt_feature_matrices[(pca_dims, hist_factor, extra_pca_dims)]
    train_split_features     = feature_matrix[train_idx]
    val_split_features       = feature_matrix[val_idx]
    train_split_labels       = train_labels[train_idx]
    val_split_labels         = train_labels[val_idx]
    total_feature_dimensions = feature_matrix.shape[1]

    # Random Forest
    for params in SEARCH_RF:
        model = RandomForestClassifier(n_estimators=params['n_trees'], max_features='sqrt',
                                       random_state=42, n_jobs=-1)
        model.fit(train_split_features, train_split_labels)
        validation_f1 = f1_score(val_split_labels, model.predict(val_split_features), average='weighted')
        record_search_result('RandomForest', f"n_trees={params['n_trees']}",
                             json.dumps({'n_trees': params['n_trees']}),
                             pca_dims, hist_factor, extra_pca_dims, total_feature_dimensions, validation_f1)
        progress_bar.update(1)

    # XGBoost
    for params in SEARCH_XGB:
        model = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', verbosity=0,
                              n_estimators=params['n_estimators'], max_depth=params['max_depth'],
                              learning_rate=params['learning_rate'])
        model.fit(train_split_features, train_split_labels)
        validation_f1 = f1_score(val_split_labels, model.predict(val_split_features), average='weighted')
        record_search_result(
            'XGBoost',
            f"n={params['n_estimators']} depth={params['max_depth']} lr={params['learning_rate']}",
            json.dumps({'n_estimators': params['n_estimators'], 'max_depth': params['max_depth'],
                        'learning_rate': params['learning_rate']}),
            pca_dims, hist_factor, extra_pca_dims, total_feature_dimensions, validation_f1
        )
        progress_bar.update(1)

    # LightGBM
    for params in SEARCH_LGBM:
        model = LGBMClassifier(n_estimators=params['n_estimators'], num_leaves=params['num_leaves'],
                               learning_rate=params['learning_rate'], n_jobs=-1, verbose=-1)
        model.fit(train_split_features, train_split_labels)
        validation_f1 = f1_score(val_split_labels, model.predict(val_split_features), average='weighted')
        record_search_result(
            'LightGBM',
            f"n={params['n_estimators']} leaves={params['num_leaves']} lr={params['learning_rate']}",
            json.dumps({'n_estimators': params['n_estimators'], 'num_leaves': params['num_leaves'],
                        'learning_rate': params['learning_rate']}),
            pca_dims, hist_factor, extra_pca_dims, total_feature_dimensions, validation_f1
        )
        progress_bar.update(1)

    # CatBoost
    for params in SEARCH_CATBOOST:
        model = CatBoostClassifier(iterations=params['iterations'], depth=params['depth'],
                                   learning_rate=params['learning_rate'], verbose=0)
        model.fit(train_split_features, train_split_labels)
        validation_f1 = f1_score(val_split_labels, model.predict(val_split_features), average='weighted')
        record_search_result(
            'CatBoost',
            f"iters={params['iterations']} depth={params['depth']} lr={params['learning_rate']}",
            json.dumps({'iterations': params['iterations'], 'depth': params['depth'],
                        'learning_rate': params['learning_rate']}),
            pca_dims, hist_factor, extra_pca_dims, total_feature_dimensions, validation_f1
        )
        progress_bar.update(1)

# --- Pass 3: MLP (slow) ------------------------------------------------------
progress_bar.set_description('Pass 3/3 - MLP')
for pca_dims, hist_factor, extra_pca_dims in all_feature_configurations:
    feature_matrix       = prebuilt_feature_matrices[(pca_dims, hist_factor, extra_pca_dims)]
    train_split_features = feature_matrix[train_idx]
    val_split_features   = feature_matrix[val_idx]
    train_split_labels   = train_labels[train_idx]
    val_split_labels     = train_labels[val_idx]
    total_feature_dimensions = feature_matrix.shape[1]

    for params in SEARCH_MLP:
        _, metrics = train_multilayer_perceptron(
            train_split_features, train_split_labels,
            val_split_features,   val_split_labels,
            hidden_layers=params['hidden_layers'],
            activation=params['activation'],
            dropout=params['dropout'],
            learning_rate=params['learning_rate'],
            epochs=300
        )
        record_search_result(
            'MLP',
            f"layers={params['hidden_layers']} act={params['activation'].__name__} drop={params['dropout']}",
            json.dumps({'layers': str(params['hidden_layers']), 'act': params['activation'].__name__,
                        'drop': params['dropout'], 'lr': params['learning_rate']}),
            pca_dims, hist_factor, extra_pca_dims, total_feature_dimensions, metrics['f1']
        )
        progress_bar.update(1)

progress_bar.close()

# --- Save and display results ------------------------------------------------
search_results_dataframe = (
    pd.DataFrame(search_results)
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)

fine_tuned_suffix           = 'ft' if USE_FINETUNED else 'noft'
search_results_csv_filename = f'hyperparameter_search_results_{FEATURE_EXTRACTOR}_{fine_tuned_suffix}.csv'
search_results_dataframe.to_csv(search_results_csv_filename, index=False)
print(f'Saved {len(search_results_dataframe)} results to {search_results_csv_filename}')

W = 112
print(f'\n{"="*W}')
print(f'  TOP 25 -- {len(search_results)} configs evaluated, ranked by weighted F1')
print(f'{"="*W}')
print(f'  {"#":<3}  {"Model":<12}  {"F1":<8}  {"Model params":<38}  {"PCA":<6}  {"Hist":<5}  {"ExtraPCA":<9}  Dims')
print(f'  {"-"*(W-2)}')
for rank, row in search_results_dataframe.head(25).iterrows():
    print(
        f'  {rank+1:<3}  '
        f'{row["model"]:<12}  '
        f'{row["f1"]:.4f}    '
        f'{str(row["params"]):<38}  '
        f'{str(row["pca_dims"]):<6}  '
        f'{row["hist_factor"]:<5}  '
        f'{row["extra_pca"]:<9}  '
        f'{row["total_dims"]}'
    )
print(f'{"="*W}')
print('\nFull results in search_results_dataframe')

## ERROR ANALYSIS
Re-train the top-5 best models (one per model type) on the training split and evaluate on the validation split.
Uses the same feature config (PCA dims, histogram factor, extra-feature PCA) that each model had during search.
Outputs two CSVs: images each model got **wrong** and images each model got **right**.

In [ ]:
import json
import ast

fine_tuned_suffix           = 'ft' if USE_FINETUNED else 'noft'
search_results_csv_path     = f'hyperparameter_search_results_{FEATURE_EXTRACTOR}_{fine_tuned_suffix}.csv'
wrong_predictions_csv_path  = f'wrong_predictions_{FEATURE_EXTRACTOR}_{fine_tuned_suffix}.csv'
correct_predictions_csv_path = f'correct_predictions_{FEATURE_EXTRACTOR}_{fine_tuned_suffix}.csv'

# Load search results and pick the best config for each unique model type
all_search_results_dataframe = pd.read_csv(search_results_csv_path)
top_five_best_models = (
    all_search_results_dataframe
    .sort_values('f1', ascending=False)
    .drop_duplicates(subset=['model'])
    .head(5)
    .reset_index(drop=True)
)
print('Top 5 models (one per type):')
for _, row in top_five_best_models.iterrows():
    print(f'  {row["model"]:<14}  search f1={row["f1"]:.4f}  {row["params"]}')
print()

# val split: the same first-fold of StratifiedKFold used in HYPER PARAM SEARCH
# train_idx, val_idx, and build_combined_features() are all available from that cell
index_to_class_name  = {v: k for k, v in train_data['class_to_index'].items()}
validation_image_ids = [
    os.path.splitext(os.path.basename(train_data['paths'][i]))[0]
    for i in val_idx
]
validation_true_labels = train_labels[val_idx]


def reconstruct_classifier_from_params(model_name, model_params):
    # Reconstruct a sklearn/lgbm/catboost model from its params dict.
    if model_name == 'LogReg':
        return LogisticRegression(C=float(model_params.get('C', 1.0)), max_iter=2000)
    elif model_name == 'SVM':
        extra_kwargs = {k: v for k, v in model_params.items() if k != 'kernel'}
        return SVC(kernel=model_params.get('kernel', 'rbf'), decision_function_shape='ovr', **extra_kwargs)
    elif model_name == 'kNN':
        return KNeighborsClassifier(
            n_neighbors=int(model_params.get('k', 10)),
            metric=model_params.get('metric', 'cosine'),
            n_jobs=-1
        )
    elif model_name == 'RandomForest':
        return RandomForestClassifier(
            n_estimators=int(model_params.get('n_trees', 300)),
            max_features='sqrt', random_state=42, n_jobs=-1
        )
    elif model_name == 'XGBoost':
        return XGBClassifier(
            objective='multi:softmax', eval_metric='mlogloss', verbosity=0,
            n_estimators=int(model_params.get('n_estimators', 100)),
            max_depth=int(model_params.get('max_depth', 6)),
            learning_rate=float(model_params.get('learning_rate', 0.1))
        )
    elif model_name == 'LightGBM':
        return LGBMClassifier(
            n_estimators=int(model_params.get('n_estimators', 500)),
            num_leaves=int(model_params.get('num_leaves', 63)),
            learning_rate=float(model_params.get('learning_rate', 0.05)),
            n_jobs=-1, verbose=-1
        )
    elif model_name == 'CatBoost':
        return CatBoostClassifier(
            iterations=int(model_params.get('iterations', 500)),
            depth=int(model_params.get('depth', 6)),
            learning_rate=float(model_params.get('learning_rate', 0.05)),
            verbose=0
        )
    return None  # MLP handled separately below


wrong_prediction_rows   = []
correct_prediction_rows = []

for _, row in top_five_best_models.iterrows():
    params_json_string = str(row.get('params_json', '{}'))
    model_params       = json.loads(params_json_string) if params_json_string and params_json_string not in ('{}', 'nan') else {}

    # Rebuild the exact feature matrix for this model's original search config
    pca_dimensions          = None if str(row['pca_dims']) == 'None' else int(row['pca_dims'])
    histogram_factor        = int(row['hist_factor'])
    extra_pca_dimensions    = None if str(row['extra_pca']) == 'None' else int(row['extra_pca'])
    feature_matrix          = build_combined_features(pca_dimensions, histogram_factor, extra_pca_dimensions)
    training_features_split = feature_matrix[train_idx]
    validation_features_split = feature_matrix[val_idx]

    if row['model'] == 'MLP':
        # MLP needs to be re-trained (non-deterministic, but same architecture)
        hidden_layer_sizes   = ast.literal_eval(model_params.get('layers', '(128, 64)'))
        activation_class_map = {'GELU': nn.GELU, 'ReLU': nn.ReLU, 'SiLU': nn.SiLU}
        activation_class     = activation_class_map.get(model_params.get('act', 'GELU'), nn.GELU)
        dropout_rate         = float(model_params.get('drop', 0.2))
        learning_rate        = float(model_params.get('lr', 1e-3))

        trained_mlp_model, mlp_fold_metrics = train_multilayer_perceptron(
            training_features_split, train_labels[train_idx],
            validation_features_split, validation_true_labels,
            hidden_layers=hidden_layer_sizes, activation=activation_class,
            dropout=dropout_rate, learning_rate=learning_rate, epochs=300
        )
        trained_mlp_model.eval()
        with torch.no_grad():
            mlp_inference_device = next(trained_mlp_model.parameters()).device
            predicted_labels     = trained_mlp_model(
                torch.tensor(validation_features_split.astype(np.float32)).to(mlp_inference_device)
            ).argmax(dim=1).cpu().numpy()
        validation_f1_score = mlp_fold_metrics['f1']
    else:
        trained_model = reconstruct_classifier_from_params(row['model'], model_params)
        if trained_model is None:
            print(f'  [skip] {row["model"]}')
            continue
        trained_model.fit(training_features_split, train_labels[train_idx])
        predicted_labels    = np.asarray(trained_model.predict(validation_features_split), int)
        validation_f1_score = f1_score(validation_true_labels, predicted_labels, average='weighted')

    print(f'  {row["model"]:<14}  retrained f1={validation_f1_score:.4f}')

    for image_id_str, predicted_label_index, true_label_index in zip(
        validation_image_ids, predicted_labels, validation_true_labels
    ):
        true_class_name      = index_to_class_name[int(true_label_index)]
        predicted_class_name = index_to_class_name[int(predicted_label_index)]
        prediction_record = {
            'image_id':        image_id_str,
            'true_class':      true_class_name,
            'predicted_class': predicted_class_name,
            'model_name':      row['model'],
            'model_params':    row['params'],
            'model_val_f1':    round(validation_f1_score, 4),
        }
        if predicted_class_name == true_class_name:
            correct_prediction_rows.append(prediction_record)
        else:
            wrong_prediction_rows.append(prediction_record)

pd.DataFrame(wrong_prediction_rows).sort_values(['image_id', 'model_name']).to_csv(
    wrong_predictions_csv_path, index=False
)
pd.DataFrame(correct_prediction_rows).sort_values(['image_id', 'model_name']).to_csv(
    correct_predictions_csv_path, index=False
)
print(f'\nWrong predictions  : {len(wrong_prediction_rows):>5}  ->  {wrong_predictions_csv_path}')
print(f'Correct predictions: {len(correct_prediction_rows):>5}  ->  {correct_predictions_csv_path}')

## SUBMISSION

In [ ]:
# --- Settings ----------------------------------------------------------------
# Choose the best model and feature config from the search above
SUBMISSION_MODEL = 'linear'   # 'linear' | 'svm' | 'rf' | 'knn' | 'xgboost' | 'lgbm' | 'catboost' | 'mlp' | 'stack'

# Feature engineering settings to use for submission
# (copy from the best row in the search results)
SUBMIT_PCA_DIMS    = n_99
SUBMIT_HIST_FACTOR = 1
SUBMIT_EXTRA_PCA   = None

# Model hyperparameters
LOGISTIC_REGRESSION_STRENGTH = 0.1
SVM_SUBMISSION_PARAMS        = {'kernel': 'linear', 'C': 1}
RANDOM_FOREST_NUM_TREES      = 300
KNN_NEAREST_NEIGHBOURS       = 10
KNN_DISTANCE_METRIC          = 'cosine'
XGBOOST_SUBMISSION_PARAMS    = {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1}
LIGHTGBM_SUBMISSION_PARAMS   = {'n_estimators': 500, 'num_leaves': 63, 'learning_rate': 0.05}
CATBOOST_SUBMISSION_PARAMS   = {'iterations': 500, 'depth': 6, 'learning_rate': 0.05}
MLP_HIDDEN_LAYERS            = (128, 64)
MLP_DROPOUT_RATE             = 0.2
MLP_TRAINING_EPOCHS          = 300
MLP_LEARNING_RATE            = 1e-3
# -----------------------------------------------------------------------------

def build_submission_features(pca_dims, hist_factor, extra_pca_dims):
    # Build feature matrices for both train and test using the full datasets.
    if USE_PCA:
        pca_reducer           = PCA(n_components=pca_dims, random_state=42)
        combined_train_features = pca_reducer.fit_transform(raw_train_features).astype(np.float64)
        combined_test_features  = pca_reducer.transform(raw_test_features).astype(np.float64)
    else:
        combined_train_features = raw_train_features.astype(np.float64)
        combined_test_features  = raw_test_features.astype(np.float64)

    if USE_COLOR_HISTOGRAM:
        col_train = load_csv(r'Data\task2_data\color_histogram.csv', train_image_ids)
        col_test  = load_csv(r'Data\task2_data\color_histogram.csv', test_image_ids)
        col_train = reduce_hist(col_train, hist_factor)
        col_test  = reduce_hist(col_test,  hist_factor)
        color_histogram_scaler  = StandardScaler().fit(col_train)
        combined_train_features = np.hstack([combined_train_features, color_histogram_scaler.transform(col_train)])
        combined_test_features  = np.hstack([combined_test_features,  color_histogram_scaler.transform(col_test)])

    if USE_HOG_FEATURES:
        hog_train = load_csv(r'Data\task2_data\hog_pca.csv', train_image_ids)
        hog_test  = load_csv(r'Data\task2_data\hog_pca.csv', test_image_ids)
        combined_train_features = np.hstack([combined_train_features, hog_train.astype(np.float64)])
        combined_test_features  = np.hstack([combined_test_features,  hog_test.astype(np.float64)])

    if USE_EXTRA_FEATURES:
        extra_train = load_csv(r'Data\task2_data\additional_features.csv', train_image_ids)
        extra_test  = load_csv(r'Data\task2_data\additional_features.csv', test_image_ids)
        extra_features_scaler       = StandardScaler().fit(extra_train)
        extra_train_features_scaled = extra_features_scaler.transform(extra_train)
        extra_test_features_scaled  = extra_features_scaler.transform(extra_test)
        if extra_pca_dims is not None:
            extra_features_pca_reducer  = PCA(n_components=extra_pca_dims, random_state=42)
            extra_train_features_scaled = extra_features_pca_reducer.fit_transform(extra_train_features_scaled)
            extra_test_features_scaled  = extra_features_pca_reducer.transform(extra_test_features_scaled)
        combined_train_features = np.hstack([combined_train_features, extra_train_features_scaled])
        combined_test_features  = np.hstack([combined_test_features,  extra_test_features_scaled])

    return combined_train_features, combined_test_features


all_train_features, submit_test_features = build_submission_features(
    SUBMIT_PCA_DIMS, SUBMIT_HIST_FACTOR, SUBMIT_EXTRA_PCA
)
all_labels = np.asarray(train_labels, np.int64)

print(f'Training {SUBMISSION_MODEL} on {len(all_train_features)} samples, '
      f'{all_train_features.shape[1]}-d features...')

if SUBMISSION_MODEL == 'linear':
    model = LogisticRegression(C=LOGISTIC_REGRESSION_STRENGTH, max_iter=2000)
    model.fit(all_train_features, all_labels)
    predictions = model.predict(submit_test_features)

elif SUBMISSION_MODEL == 'svm':
    model = SVC(decision_function_shape='ovr', **SVM_SUBMISSION_PARAMS)
    model.fit(all_train_features, all_labels)
    predictions = model.predict(submit_test_features)

elif SUBMISSION_MODEL == 'rf':
    model = RandomForestClassifier(n_estimators=RANDOM_FOREST_NUM_TREES, max_features='sqrt',
                                   random_state=42, n_jobs=-1)
    model.fit(all_train_features, all_labels)
    predictions = model.predict(submit_test_features)

elif SUBMISSION_MODEL == 'knn':
    model = KNeighborsClassifier(n_neighbors=KNN_NEAREST_NEIGHBOURS, metric=KNN_DISTANCE_METRIC, n_jobs=-1)
    model.fit(all_train_features, all_labels)
    predictions = model.predict(submit_test_features)

elif SUBMISSION_MODEL == 'xgboost':
    model = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', verbosity=0,
                          **XGBOOST_SUBMISSION_PARAMS)
    model.fit(all_train_features, all_labels)
    predictions = model.predict(submit_test_features)

elif SUBMISSION_MODEL == 'lgbm':
    model = LGBMClassifier(**LIGHTGBM_SUBMISSION_PARAMS, n_jobs=-1, verbose=-1)
    model.fit(all_train_features, all_labels)
    predictions = model.predict(submit_test_features)

elif SUBMISSION_MODEL == 'catboost':
    model = CatBoostClassifier(**CATBOOST_SUBMISSION_PARAMS, verbose=0)
    model.fit(all_train_features, all_labels)
    predictions = model.predict(submit_test_features).astype(int)

elif SUBMISSION_MODEL == 'stack':
    # Hold out 5% for the stacker's internal CV -- prevents over-fitting
    stack_train_features, stack_val_features, stack_train_labels, stack_val_labels = train_test_split(
        all_train_features, all_labels, test_size=0.05, random_state=42, stratify=all_labels
    )
    model, _ = train_stacking_ensemble(
        stack_train_features, stack_train_labels,
        stack_val_features,   stack_val_labels
    )
    predictions = model.predict(submit_test_features)

elif SUBMISSION_MODEL == 'mlp':
    # Hold out 5% for early stopping -- MLP needs a validation signal during training
    mlp_train_features, mlp_val_features, mlp_train_labels, mlp_val_labels = train_test_split(
        all_train_features, all_labels, test_size=0.05, random_state=42, stratify=all_labels
    )
    model, _ = train_multilayer_perceptron(
        mlp_train_features, mlp_train_labels,
        mlp_val_features,   mlp_val_labels,
        hidden_layers=MLP_HIDDEN_LAYERS,
        dropout=MLP_DROPOUT_RATE,
        epochs=MLP_TRAINING_EPOCHS,
        learning_rate=MLP_LEARNING_RATE
    )
    model.eval()
    with torch.no_grad():
        mlp_inference_device = next(model.parameters()).device
        predictions = model(
            torch.tensor(np.asarray(submit_test_features, np.float32)).to(mlp_inference_device)
        ).argmax(dim=1).cpu().numpy()

else:
    raise ValueError(f'Unknown model: {SUBMISSION_MODEL}')

image_ids  = [os.path.splitext(os.path.basename(path))[0] for path in test_data['paths']]
submission = pd.DataFrame({'image_id': image_ids, 'class_id': predictions})
submission.to_csv('t2_submission.csv', index=False)
print(f'Saved {len(submission)} predictions to t2_submission.csv')